# Content-based Filtering

In [90]:
# Import libraries
import json
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [91]:
# Load final merged dataset
df_final = pd.read_csv("../data/processed/restaurants_data_merged_final.csv")

df_final.head()

,name,price,rating,review_count,cuisine,address,display_phone,comments
0,123 Burger Shot Beer,1,3.0,1000.0,"['american', 'sportsbars', 'tradamerican', 'ch...","738 10th Ave, Hells Kitchen, NY 10019",(212) 315-0123,NaN
1,One Stop Patty Shop,1,4.0,40.0,"['bakery', 'caribbean', 'breakfast_brunch']","1708 Amsterdam Ave, Harlem, NY 10031",(212) 491-7466,NaN
2,108 Food Dried Hot Pot,2,3.5,139.0,"['chinese', 'hotpot']","2794 Broadway, East Harlem, NY 10025",(917) 675-6878,NaN
3,Cookshop,2,4.0,1000.0,"['american', 'newamerican', 'breakfast_brunch'...","156 10th Ave, Midtown West, NY 10011",(212) 924-4440,NaN
4,11 Hanover Greek,3,4.0,122.0,"['greek', 'seafood', 'wine_bars']","11 Hanover Sq, Tribeca, NY 10005",(212) 785-4000,NaN


In [92]:
# Extract neighborhood from address
def extract_neighborhood(address):
    try:
        parts = address.split(",")
        if len(parts) >= 2:
            return parts[1].strip()
        return ""
    except:
        return ""

df_final["neighborhood"] = df_final["address"].apply(extract_neighborhood)

In [93]:
# Convert price and rating to text
def price_to_text(p):
    if p == 1:
        return "cheap"
    elif p == 2:
        return "moderate"
    elif p >= 3:
        return "expensive"
    return ""

def rating_to_text(r):
    if r >= 4:
        return "high_rating"
    elif r >= 3:
        return "medium_rating"
    else:
        return "low_rating"

df_final["price_text"] = df_final["price"].apply(price_to_text)
df_final["rating_text"] = df_final["rating"].apply(rating_to_text)

In [94]:
# Fill missing comments with empty string
df_final["comments"] = df_final["comments"].fillna("")

In [95]:
# Clean cuisine column 
def clean_cuisine(c):
    try:
        c_list = ast.literal_eval(c)
        return " ".join(c_list)
    except:
        return ""

df_final["cuisine_clean"] = df_final["cuisine"].apply(clean_cuisine)

# optional improvement
df_final["cuisine_clean"] = df_final["cuisine_clean"].str.replace("_", " ")

In [96]:
# Create combined features for content-based filtering
df_final["features"] = (
    df_final["cuisine_clean"] + " " +
    df_final["neighborhood"] + " " +
    df_final["price_text"] + " " +
    df_final["rating_text"] + " " +
    df_final["comments"]
)

In [97]:
# Vectorize features using TF-IDF
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(df_final["features"])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [121]:
# Create a mapping of restaurant names to indices
indices = pd.Series(df_final.index, index=df_final["name"]).drop_duplicates()

# Recommendation function
def recommend_restaurants(name, top_n=5):
    
    if name not in indices:
        return "Restaurant not found"
    
    idx = indices[name]
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+20]
    
    results = []
    
    for i, sim_score in sim_scores:
        row = df_final.iloc[i]
        
        final_score = (
            sim_score * 0.6 +
            (row["rating"] / 5) * 0.3 +
            (row["review_count"] / df_final["review_count"].max()) * 0.1
        )
        
        results.append((i, final_score))
    
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_n]
    
    return df_final.iloc[[i[0] for i in results]][["name","cuisine_clean","price","rating"]]

In [122]:
df_final["features"] = df_final["features"].str.lower()
df_final["features"] = df_final["features"].str.replace("_", " ")

In [123]:
# Test the recommendation function with top 3 recommendations
recommend_restaurants("Ming's Caffe", top_n=3)

,name,cuisine_clean,price,rating
2748,S Wan Cafe 洋紫荆,chinese hkcafe,1,4.5
1584,Joe’s Steam Rice Roll,chinese hkcafe hotdogs cantonese,1,4.0
1681,King's Kitchen,chinese,1,4.0


In [124]:
# Test the recommendation function with top 5 recommendations
recommend_restaurants("The Thirsty Koala", top_n=5)

,name,cuisine_clean,price,rating
3918,Trattoria L'incontro,italian vegetarian friendly vegan options,2,4.5
4361,Burke & Wills,australian,2,5.0
2734,Ruby's Cafe,australian bars breakfast brunch,2,5.0
4013,Martha's Country Bakery,bakeries cafe,2,4.5
3912,Osteria 166,italian vegetarian friendly vegan options,2,4.5


In [125]:
# Test the recommendation function with top 10 recommendations
recommend_restaurants("212 Steakhouse", top_n=10)

,name,cuisine_clean,price,rating
1044,Empire Steak House,american steak wine bars seafood,3,4.0
2522,Pietro's,italian steak wine bars,3,4.0
336,Benjamin Steakhouse,steak seafood,4,4.0
3191,Tempura Matsui,japanese seafood,4,4.5
2711,Rocco Steakhouse,steak tradamerican seafood,3,4.5
3644,Wokuni,japanese seafood,3,4.0
2999,Steak 'N Lobster,steak bars seafood,2,4.0
3505,Tuttles,american bars seafood,2,3.5
458,Bowery Meat Company,american steak wine bars,3,4.0
1913,Luke's Lobster Midtown East,seafood,2,4.0


# Evaluation

In [126]:
# Function to calculate cuisine overlap score for a given restaurant
def cuisine_overlap_score(base_name, top_n=5):
    
    recs = recommend_restaurants(base_name, top_n)
    
    base_cuisine = df_final[df_final["name"] == base_name]["cuisine_clean"].values[0]
    base_set = set(base_cuisine.split())
    
    scores = []
    
    for _, row in recs.iterrows():
        rec_set = set(row["cuisine_clean"].split())
        
        if len(base_set) == 0:
            scores.append(0)
        else:
            overlap = len(base_set & rec_set) / len(base_set)
            scores.append(overlap)
    
    return np.mean(scores)

In [119]:
# Test the cusine overlap score function
print("Ming's Caffe cuisine overlap score:", cuisine_overlap_score("Ming's Caffe", top_n=3))
print("The Thirsty Koala cuisine overlap score:", cuisine_overlap_score("The Thirsty Koala", top_n=5))
print("212 Steakhouse cuisine overlap score:", cuisine_overlap_score("212 Steakhouse", top_n=10))

Ming's Caffe cuisine overlap score: 0.8333333333333334
The Thirsty Koala cuisine overlap score: 0.4
212 Steakhouse cuisine overlap score: 0.55


In [127]:
# Function to calculate average rating score for recommended restaurants
def avg_rating_score(base_name, top_n=5):
    recs = recommend_restaurants(base_name, top_n)
    return recs["rating"].mean()

print("Ming's Caffe average rating score:", avg_rating_score("Ming's Caffe", top_n=3))
print("The Thirsty Koala average rating score:", avg_rating_score("The Thirsty Koala", top_n=5))
print("212 Steakhouse average rating score:", avg_rating_score("212 Steakhouse", top_n=10))

Ming's Caffe average rating score: 4.166666666666667
The Thirsty Koala average rating score: 4.7
212 Steakhouse average rating score: 4.05


In [128]:
# Function to calculate diversity score for recommended restaurants
def diversity_score(base_name, top_n=5):
    
    recs = recommend_restaurants(base_name, top_n)
    
    cuisines = recs["cuisine_clean"].tolist()
    unique = len(set(cuisines))
    
    return unique / len(cuisines)

print("Ming's Caffe diversity score:", diversity_score("Ming's Caffe", top_n=3))
print("The Thirsty Koala diversity score:", diversity_score("The Thirsty Koala", top_n=5))
print("212 Steakhouse diversity score:", diversity_score("212 Steakhouse", top_n=10))

Ming's Caffe diversity score: 1.0
The Thirsty Koala diversity score: 0.8
212 Steakhouse diversity score: 0.9


| Metric             | Good Value |
| ------------------ | ---------- |
| Cuisine similarity | 0.4 – 0.8  |
| Avg rating         | > 3.8      |
| Diversity          | 0.4 – 0.7  |


In [130]:
# Function to calculate precision@k for recommended restaurants
def precision_at_k(base_name, k=5):
    
    recs = recommend_restaurants(base_name, k)
    
    base_cuisine = df_final[df_final["name"] == base_name]["cuisine_clean"].values[0]
    
    relevant = 0
    
    for _, row in recs.iterrows():
        if any(c in row["cuisine_clean"] for c in base_cuisine.split()):
            relevant += 1
    
    return relevant / k

| Precision@K | Meaning                                    |
| ----------- | ------------------------------------------ |
| 1.0         | Perfect (all recommendations are relevant) |
| 0.8         | Very strong                                |
| 0.6         | Good                                       |
| 0.4         | Weak                                       |
| < 0.3       | Poor                                       |


In [131]:
# Precision@k for top 5 recommendations
print("Ming's Caffe precision@5:", precision_at_k("Ming's Caffe", k=5))
print("The Thirsty Koala precision@5:", precision_at_k("The Thirsty Koala", k=5))
print("212 Steakhouse precision@5:", precision_at_k("212 Steakhouse", k=5))

Ming's Caffe precision@5: 1.0
The Thirsty Koala precision@5: 0.8
212 Steakhouse precision@5: 1.0
